# Evaluating LLM Applications

This notebook is designed for interviews and practical deployment discussions. It covers how to evaluate LLM systems with manual rubrics, golden datasets, regression tests, automated metrics, LLM-as-judge workflows, and online experiments. The emphasis is reproducible quality control, not one-time demos. By the end, you should be able to explain an end-to-end evaluation strategy for a production LLM application.


## 1. Evaluation as Product Infrastructure

Evaluation is not a final checklist; it is continuous infrastructure that guides release decisions. LLM systems change frequently through prompt updates, retrieval modifications, model upgrades, and policy tuning. Without stable evaluation loops, quality drift is inevitable and often invisible until users are impacted.

Interview-ready framing: "If you cannot measure it repeatedly, you cannot safely ship it repeatedly." This line communicates engineering discipline.


## 2. Defining Quality Dimensions Clearly

Quality should be decomposed into explicit dimensions: correctness, faithfulness, relevance, instruction adherence, safety, latency, and cost. Different applications prioritize different dimensions, but each must be defined with scoring criteria and acceptable thresholds.

Ambiguous dimensions lead to evaluator disagreement and noisy metrics. In interviews, mention score anchors and adjudication rules to show you understand evaluation reliability, not just evaluation volume.


## 3. Manual Rubrics and Scoring Anchors

Manual rubrics translate qualitative expectations into repeatable scoring behavior. For each dimension, define score bands with examples so different reviewers interpret standards consistently. Rubrics are critical during early development because they reveal failure modes that automatic metrics often miss.

A mature process periodically recalibrates rubric definitions based on disagreement analysis and incident reviews.


In [ ]:
import numpy as np

# Weighted rubric scorer for a single response
WEIGHTS = {
    "correctness": 0.30,
    "faithfulness": 0.25,
    "relevance": 0.20,
    "clarity": 0.10,
    "safety": 0.15,
}

def rubric_score(item):
    return sum(item[k] * w for k, w in WEIGHTS.items())

example = {
    "correctness": 0.88,
    "faithfulness": 0.81,
    "relevance": 0.91,
    "clarity": 0.86,
    "safety": 0.98,
}
print("Composite rubric score:", round(rubric_score(example), 3))


## 4. Golden Datasets and Coverage Strategy

Golden datasets should represent realistic traffic, high-impact tasks, and known edge cases. A small but carefully curated set can outperform a large random set for regression detection. Include adversarial prompts, ambiguity cases, and policy-sensitive examples to avoid benchmark blind spots.

In interviews, say that golden sets should be versioned and tagged by scenario type. This supports targeted analysis when regressions appear in specific categories.


## 5. Regression Testing for Every Change

Every prompt or model change should be evaluated against the same baseline dataset to detect quality deltas. Aggregate score improvement is not sufficient; you must inspect category-level regressions and minimum-score failures in high-risk slices.

Release gates should fail closed when critical metrics drop below thresholds or when required evaluation data is missing.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(21)
categories = ["FAQ", "Policy QA", "Summarization", "Safety"]
baseline = np.array([0.82, 0.76, 0.84, 0.93])
candidate = np.array([0.85, 0.73, 0.87, 0.92])

delta = candidate - baseline
print("Category deltas:")
for c, d in zip(categories, delta):
    print(f"{c:14s}: {d:+.3f}")

x = np.arange(len(categories))
w = 0.35
plt.figure(figsize=(9, 4))
plt.bar(x - w/2, baseline, width=w, label="Baseline")
plt.bar(x + w/2, candidate, width=w, label="Candidate")
plt.xticks(x, categories)
plt.ylim(0.6, 1.0)
plt.title("Regression Comparison by Category")
plt.ylabel("Score")
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.show()


## 6. Core Metrics: BLEU, ROUGE, Perplexity, Faithfulness

BLEU and ROUGE are overlap-based metrics useful for some constrained generation tasks, but they can miss semantic quality and factual grounding. Perplexity measures how probable text is under a model distribution, not whether it is correct for user intent. Faithfulness measures whether claims are supported by provided context and is often more relevant in grounded QA systems.

Interview tip: present metrics as complementary signals. No single metric should be the sole release criterion.


In [ ]:
import numpy as np
from collections import Counter

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def rouge1_recall(reference, candidate):
    r = Counter(reference.split())
    c = Counter(candidate.split())
    overlap = sum((r & c).values())
    return overlap / max(1, sum(r.values()))

ref = "the model should answer only using retrieved evidence"
cand = "the model should answer using retrieved evidence and cite sources"
print("ROUGE-1 recall (toy):", round(rouge1_recall(ref, cand), 3))

# Toy perplexity from synthetic token probabilities
probs = np.array([0.30, 0.15, 0.40, 0.25, 0.10])
perplexity = np.exp(-np.mean(np.log(probs)))
print("Perplexity (toy):", round(float(perplexity), 3))


## 7. Faithfulness and Groundedness Testing

For retrieval-based systems, faithfulness is often the make-or-break metric. A response can be fluent and relevant yet still ungrounded if claims are not supported by context passages. Faithfulness tests should include claim-evidence alignment checks and contradiction detection.

In interviews, emphasize that groundedness evaluation requires storing retrieved context with each output so evidence can be audited later.


## 8. LLM-as-Judge: Strengths

LLM-as-judge can scale pairwise and rubric-based comparisons quickly, reducing manual effort and improving iteration speed. It is especially useful for ranking candidate prompts, selecting response variants, and triaging large evaluation batches before human review.

Used carefully, judge models accelerate experimentation and reduce bottlenecks in evaluation pipelines.


## 9. LLM-as-Judge: Risks and Controls

Judge models can exhibit bias toward verbosity, style, or familiar phrasing. They may also share weaknesses with the generation model, leading to correlated blind spots. Without calibration against human-labeled subsets, judge scores can become misleadingly precise.

Controls include periodic human audits, judge-prompt standardization, inter-judge agreement checks, and adversarial test slices.


In [ ]:
import numpy as np

# Toy LLM-as-judge simulation: style bias effect
np.random.seed(5)
true_quality = np.array([0.82, 0.79, 0.75, 0.88, 0.71])
verbosity_bonus = np.array([0.04, 0.00, 0.06, 0.03, 0.05])
judge_score = np.clip(true_quality + verbosity_bonus, 0, 1)

for i, (t, j) in enumerate(zip(true_quality, judge_score), start=1):
    print(f"Response {i}: true={t:.2f}, judge={j:.2f}, bias={j-t:+.2f}")


## 10. Human Evaluation and Adjudication

Human reviewers remain essential for nuanced criteria such as tone, policy compliance in ambiguous contexts, and domain-specific appropriateness. Use double-review and adjudication for contentious samples to improve label quality.

Inter-rater agreement should be tracked as a health metric for rubric clarity and evaluator calibration.


## 11. Offline vs Online Evaluation

Offline evaluation is fast, controlled, and reproducible, making it ideal for iteration and regression detection. Online evaluation captures real user behavior, long-tail prompts, and product-level outcomes such as satisfaction, retention, and task success.

Interview best answer: "Offline and online are complementary; offline catches known failures early, online validates real-world impact and interaction effects."


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Toy relationship between offline score and online success
offline = np.array([0.70, 0.74, 0.78, 0.81, 0.84, 0.86])
online = np.array([0.62, 0.66, 0.67, 0.72, 0.74, 0.73])

corr = np.corrcoef(offline, online)[0, 1]
print("Offline-online correlation (toy):", round(float(corr), 3))

plt.figure(figsize=(6, 5))
plt.scatter(offline, online, s=70)
for x, y, i in zip(offline, online, range(1, len(offline)+1)):
    plt.text(x + 0.002, y + 0.002, f"v{i}", fontsize=9)
plt.xlabel("Offline eval score")
plt.ylabel("Online success metric")
plt.title("Offline vs Online Alignment")
plt.grid(alpha=0.25)
plt.show()


## 12. A/B Testing for LLM Applications

A/B tests evaluate end-to-end product impact across user populations. Key metrics include task completion, fallback rate, escalation rate, user satisfaction, and incident frequency. Statistical guardrails are crucial because LLM outputs can have high variance.

In interviews, mention that safety and policy regressions should have hard-stop triggers even if engagement metrics improve.


## 13. Error Analysis and Iterative Improvement

After scoring, cluster failures into categories such as hallucination, omission, instruction drift, formatting errors, and unsafe outputs. Prioritize fixes by impact and frequency. This transforms evaluation from reporting into improvement.

Iteration should be controlled: change one major variable at a time where possible and rerun the benchmark for causal clarity.


## 14. Batch Evaluation Pipeline Design

A practical pipeline includes dataset versioning, deterministic execution configs, score aggregation by slice, artifact logging, and dashboard-based trend monitoring. Store prompt, retrieved context, output, and metric details for every sample to support auditability.

Interviewers value candidates who connect metrics to deployment gates and incident response workflows.


In [ ]:
import numpy as np

# Batch evaluation on synthetic samples
np.random.seed(8)
n = 40
scores = {
    "correctness": np.random.uniform(0.55, 0.95, n),
    "faithfulness": np.random.uniform(0.50, 0.96, n),
    "relevance": np.random.uniform(0.60, 0.97, n),
    "safety": np.random.uniform(0.85, 1.00, n),
}
composite = (
    0.30 * scores["correctness"]
    + 0.30 * scores["faithfulness"]
    + 0.20 * scores["relevance"]
    + 0.20 * scores["safety"]
)

print("Batch mean composite:", round(float(composite.mean()), 3))
print("Batch p10 composite:", round(float(np.percentile(composite, 10)), 3))
print("Fail count (<0.72):", int((composite < 0.72).sum()))


## 15. Interview Focus: Detailed Q&A

1. **Q: How do you evaluate an LLM application end-to-end?**  
   **A:** Start with a rubric aligned to product goals, build a versioned golden dataset with edge cases, run offline regression metrics by slice, then validate in online experiments with user-centric and safety metrics.

2. **Q: Why are manual rubrics still needed?**  
   **A:** Automated metrics miss nuance and policy context. Rubrics define quality intent and provide calibration data for automation.

3. **Q: What is the role of golden datasets?**  
   **A:** They provide repeatable baselines for detecting regressions and comparing variants fairly.

4. **Q: BLEU/ROUGE vs faithfulness - when to use what?**  
   **A:** BLEU/ROUGE help in overlap-sensitive tasks; faithfulness is critical for grounded QA where claim support matters.

5. **Q: What does perplexity tell you?**  
   **A:** It reflects sequence likelihood under a model, not direct task correctness or factuality.

6. **Q: Is LLM-as-judge reliable enough alone?**  
   **A:** No. It is scalable but biased; calibrate with human labels and monitor agreement drift.

7. **Q: Offline or online evaluation - which is more important?**  
   **A:** Both. Offline is faster and controlled; online confirms real user impact and long-tail behavior.

8. **Q: What should block a release?**  
   **A:** Regression in critical slices, safety threshold failures, or missing required evaluation evidence.

9. **Q: How do you make evaluation actionable?**  
   **A:** Tie score deltas to release gates, run error clustering, prioritize fixes, and rerun benchmarks for measurable improvement.


## 16. Summary

Reliable LLM evaluation blends manual judgment, automated metrics, and operational experimentation. Rubrics and golden datasets define quality; regression tests and release gates enforce it; human review and online testing validate real-world behavior.

In interviews, emphasize that evaluation is continuous infrastructure. The strongest teams treat model quality like software quality: measurable, monitored, and tightly coupled to deployment decisions.


## 17. Key Takeaways

- Define quality dimensions explicitly and score them consistently.
- Use golden datasets and regression tests for every significant change.
- Combine BLEU/ROUGE/perplexity with faithfulness and safety metrics.
- Use LLM-as-judge for scale, but calibrate with human evaluation.
- Integrate offline and online evaluation into release governance.
